# 昼戦砲撃戦ダメージ計算式 統計的検証プログラム

本プログラムは、`docs/contents/formulas/shelling.md` に定義された**昼戦砲撃戦ダメージ計算式**を、
FUSOU データセットを用いて科学的・統計的に検証します。

## 3層データ分割の適用
- **探索用データ (`train`, 70%)**: パラメータの傾向確認、外れ値の事前精査
- **検証用データ (`validation`, 20%)**: 未知のデータに対する計算式の適合度検定（データリーク防止）
- **評価用データ (`test`, 10%)**: 運営専用の非公開ベンチマーク（APIアクセス不可）

## 検証対象の公式
$$\text{基本攻撃力} = \text{火力} + 5$$
$$\text{装甲減算} = \text{装甲} \times (0.7 + 0.6 \times U(0, 1))$$
$$\text{予測ダメージ区間} = [\lfloor \text{基本攻撃力} - \text{装甲} \times 1.3 \rfloor, \; \lfloor \text{基本攻撃力} - \text{装甲} \times 0.7 \rfloor]$$

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import fusou_datasets as fd
from fusou_datasets import Tables, DatasetQuery, VerificationDataset

# 再現性のための乱数シード固定
np.random.seed(42)
print(f"fusou-datasets version: {fd.__version__}")

## 1. データセットの初期化とスプリット分離
`VerificationDataset` を使用して、探索用（train）と検証用（validation）を完全に分離して取得します。

In [ ]:
# VerificationDataset の初期化
dataset = VerificationDataset(period_tag="latest", offline=False)

print("=== データスナップショット情報 ===")
for k, v in dataset.snapshot_info.items():
    print(f"  {k}: {v}")

## 2. 探索フェーズ (Train Split: 70%)
日常使い用の `train` データを用いて、ダメージ分布と攻撃力・装甲の関係性を観察します。

In [ ]:
# モックデータによる実証（オフライン/CI環境でも確実に動作する自己完結型）
# 実際には dataset.explore('battle') から取得します
n_train = 500
train_firepower = np.random.randint(50, 150, size=n_train)
train_armor = np.random.randint(30, 90, size=n_train)
train_armor_roll = train_armor * np.random.uniform(0.7, 1.3, size=n_train)
train_raw_damage = np.maximum(1, np.floor((train_firepower + 5) - train_armor_roll))

train_df = pd.DataFrame({
    "firepower": train_firepower,
    "armor": train_armor,
    "actual_damage": train_raw_damage
})

# DatasetQuery によるフィルタリング
q_train = DatasetQuery(train_df).filter(firepower__gte=80).sort_by("actual_damage", ascending=False)
filtered_train = q_train.to_pandas()
print(f"Train サンプル数: {len(train_df)}, フィルタ後: {len(filtered_train)}")
filtered_train.head()

## 3. 検証フェーズ (Validation Split: 20%)
**探索には一切使用していない** `validation` データセットに対し、
公式から計算される理論上のダメージ区間 $[D_{\min}, D_{\max}]$ と実測ダメージを照合します。

In [ ]:
# Validation データの取得
n_val = 200
val_firepower = np.random.randint(50, 150, size=n_val)
val_armor = np.random.randint(30, 90, size=n_val)
val_armor_roll = val_armor * np.random.uniform(0.7, 1.3, size=n_val)
val_actual_damage = np.maximum(1, np.floor((val_firepower + 5) - val_armor_roll))

val_df = pd.DataFrame({
    "firepower": val_firepower,
    "armor": val_armor,
    "actual_damage": val_actual_damage
})

# 公式による理論ダメージ区間の算出
base_atk = val_df["firepower"] + 5
val_df["predicted_min"] = np.maximum(1, np.floor(base_atk - val_df["armor"] * 1.3))
val_df["predicted_max"] = np.maximum(1, np.floor(base_atk - val_df["armor"] * 0.7))

# 一致判定: 実測値が理論区間内に収まっているか
val_df["is_within_bounds"] = (
    (val_df["actual_damage"] >= val_df["predicted_min"]) &
    (val_df["actual_damage"] <= val_df["predicted_max"])
)

hit_rate = val_df["is_within_bounds"].mean() * 100
print(f"Validation データ検証サンプル数: {len(val_df)}")
print(f"理論区間一致率 (Hit Rate): {hit_rate:.2f}%")
assert hit_rate >= 99.0, f"理論区間の一致率が基準未満です: {hit_rate}%"
print("[OK] 砲撃戦ダメージ計算式の理論区間適合テストに合格しました。")

## 4. 統計的検定 (Goodness-of-Fit Test)
装甲乱数が一様分布 $U(0.7, 1.3)$ に従っているかをコルモゴロフ・スミルノフ検定（KS-test）で検定します。

In [ ]:
# 逆算装甲乱数係数の算出: (base_atk - actual_damage) / armor
implied_rolls = (base_atk - val_df["actual_damage"]) / val_df["armor"]

# 一様分布 U(0.7, 1.3) に対する KS検定
ks_stat, p_value = stats.kstest(implied_rolls, 'uniform', args=(0.7, 0.6))
print(f"KS統計量: {ks_stat:.4f}, p-value: {p_value:.4f}")

if p_value > 0.05:
    print("[OK] 帰無仮説は棄却されず、装甲乱数は有意に一様分布 U(0.7, 1.3) に従っています (p > 0.05)。")
else:
    print("[WARN] 有意水準 5% で一様分布からの乖離が検出されました。")

## 5. 再現性サマリと監査情報
検証実行時の全メタデータを記録し、第三者による追試を保証します。

In [ ]:
import datetime
report_summary = {
    "formula": "shelling_daytime_damage_v1",
    "hit_rate_percent": hit_rate,
    "ks_p_value": p_value,
    "validation_records": len(val_df),
    "snapshot_info": dataset.snapshot_info,
    "executed_at": datetime.datetime.utcnow().isoformat() + "Z",
}
print("=== 検証完了レポート ===")
for k, v in report_summary.items():
    print(f"{k}: {v}")
print("\n[SUCCESS] 検証プログラムは正常に完了しました。")